# Kronos SPUS Small v1 — Colab Free (T4)

Fine-tunes `NeoQuasar/Kronos-small` on SPUS daily CSVs. Tokenizer stays frozen.

**Before you run**
1. Runtime → Change runtime type → **T4 GPU**
2. On your PC: `powershell -File scripts/pack_spus_for_colab.ps1`
3. Upload `data/spus_for_colab.zip` to Drive as `MyDrive/kronos/data/spus_for_colab.zip`

Repo: https://github.com/0xboy/Kronos

## 1) GPU check

In [ ]:
import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram_gb:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise SystemExit("No GPU — Runtime → Change runtime type → T4 GPU, then reconnect.")

## 2) Mount Drive + paths

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/kronos")
DATA_DIR = DRIVE_ROOT / "data" / "spus"
DATA_ZIP = DRIVE_ROOT / "data" / "spus_for_colab.zip"
FINETUNE_DIR = DRIVE_ROOT / "finetuned"
REPO_DIR = Path("/content/Kronos")

for p in (DRIVE_ROOT / "data", FINETUNE_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT", DRIVE_ROOT)
print("DATA_DIR  ", DATA_DIR)
print("FINETUNE  ", FINETUNE_DIR)

## 3) Clone / update repo

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/0xboy/Kronos.git"
REPO_DIR = Path("/content/Kronos")

if REPO_DIR.exists():
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
print(subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())
print("cwd", os.getcwd())

## 4) Install deps

In [ ]:
# Colab already has a CUDA torch; install the rest from the repo.
%pip install -q einops==0.8.1 huggingface_hub==0.33.1 safetensors==0.6.2 tqdm pyyaml matplotlib

## 5) Unpack SPUS data onto Drive (once)

In [ ]:
import zipfile

csv_count = len(list(DATA_DIR.glob("*.csv"))) if DATA_DIR.exists() else 0
print(f"existing csv count: {csv_count}")

if csv_count >= 50:
    print("SPUS data already present — skip unzip.")
elif DATA_ZIP.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP, "r") as zf:
        zf.extractall(DATA_DIR)
    csv_count = len(list(DATA_DIR.glob("*.csv")))
    print(f"unzipped → {DATA_DIR} ({csv_count} csv)")
else:
    raise SystemExit(
        f"Missing data.\n"
        f"1) Locally: powershell -File scripts/pack_spus_for_colab.ps1\n"
        f"2) Upload zip to: {DATA_ZIP}\n"
        f"   or put CSVs directly in: {DATA_DIR}"
    )

## 6) Train (predictor only)

In [ ]:
import os
import subprocess

os.chdir(REPO_DIR / "finetune_csv")
print("cwd", os.getcwd())

# Must run from finetune_csv/ (script adds ../ to sys.path).
subprocess.check_call([
    "python", "train_sequential.py",
    "--config", "configs/config_spus_small_v1_colab.yaml",
    "--skip-tokenizer",
])

## 7) Checkpoint on Drive

In [ ]:
best = FINETUNE_DIR / "spus_small_v1" / "basemodel" / "best_model"
print("best_model dir:", best)
if best.exists():
    for p in sorted(best.iterdir()):
        if p.is_file():
            mb = p.stat().st_size / 1e6
            print(f"  {p.name:40s} {mb:8.2f} MB")
        else:
            print(f"  {p.name}/")
else:
    print("Not found yet — training may still be running or failed.")

print("\nLocal paper alias already points at spus-small-v1.")
print("After download, put files under:")
print("  finetune_csv/finetuned/spus_small_v1/basemodel/best_model/")

### Tips
- Session idle disconnects kill `/content` but **Drive checkpoints survive**.
- OOM → edit `config_spus_small_v1_colab.yaml` `batch_size: 4` and re-run cell 6.
- Resume / overwrite: set `skip_existing: true` in config or pass `--skip-existing`.